# Teams Statistics by Season

In [1]:
from pathlib import Path
import warnings
import pandas as pd
import numpy as np

All `Team*.csv` files in `../../data/nba_database`

In [2]:
DATA_DIR = "../../data/nba_database"

files = {
    "TeamHistories": f"{DATA_DIR}/TeamHistories.csv",
    "TeamStatistics": f"{DATA_DIR}/TeamStatistics.csv",
    "TeamStatisticsAdvanced": f"{DATA_DIR}/TeamStatisticsAdvanced.csv",
    "TeamStatisticsFourFactors": f"{DATA_DIR}/TeamStatisticsFourFactors.csv",
    "TeamStatisticsMisc": f"{DATA_DIR}/TeamStatisticsMisc.csv",
    "TeamStatisticsScoring": f"{DATA_DIR}/TeamStatisticsScoring.csv",
}

dfs = {name: pd.read_csv(path) for name, path in files.items()}
print("Done!")

Done!


In [3]:
df = dfs["TeamStatistics"].copy()

# gameId: PPPYYGGGGG
df["season"]      = df["gameId"].astype(str).str[1:3].astype(int) + 2000
df["season_type"] = df["gameId"].astype(str).str[0]  # 1=pre, 2=regular, 3=all-star, 4=post

print(df.groupby("season")["gameDateTimeEst"].agg(["min", "max"]))

                        min                  max
season                                          
2000    2000-10-31 19:30:00  2001-06-15 21:00:00
2001    2001-10-30 19:00:00  2002-06-12 21:00:00
2002    2002-10-29 19:30:00  2003-06-15 20:30:00
2003    2003-10-05 18:00:00  2004-06-15 21:00:00
2004    2004-10-22 19:00:00  2005-06-23 21:00:00
...                     ...                  ...
2095    1995-11-03 20:00:00  1996-06-16 20:00:00
2096    1996-11-01 19:00:00  1997-06-13 21:00:00
2097    1997-10-31 19:30:00  1998-06-14 19:30:00
2098    1999-02-05 19:00:00  1999-06-25 21:00:00
2099    1999-11-02 19:00:00  2000-06-19 21:00:00

[80 rows x 2 columns]


In [4]:
per_season = (
    df.groupby(["teamId", "teamName", "season"])
      .agg(
          games_played        = ("gameId",                    "count"),
          wins                = ("win",                       "sum"),
          losses              = ("win",                       lambda x: (x == 0).sum()),
          avg_points_scored   = ("teamScore",                 "mean"),
          avg_points_allowed  = ("opponentScore",             "mean"),
          avg_assists         = ("assists",                   "mean"),
          avg_rebounds        = ("reboundsTotal",             "mean"),
          avg_turnovers       = ("turnovers",                 "mean"),
          avg_steals          = ("steals",                    "mean"),
          avg_blocks          = ("blocks",                    "mean"),
          avg_fg_pct          = ("fieldGoalsPercentage",      "mean"),
          avg_3pt_pct         = ("threePointersPercentage",   "mean"),
          avg_ft_pct          = ("freeThrowsPercentage",      "mean"),
          avg_bench_points    = ("benchPoints",               "mean"),
          avg_paint_points    = ("pointsInThePaint",          "mean"),
          avg_fast_break_pts  = ("pointsFastBreak",           "mean"),
          avg_2nd_chance_pts  = ("pointsSecondChance",        "mean"),
          avg_biggest_lead    = ("biggestLead",               "mean"),
          avg_plus_minus      = ("plusMinusPoints",           "mean"),
      )
      .reset_index()
)

print(f"shape: {per_season.shape}")
per_season.head()

shape: (1698, 22)


,teamId,teamName,season,games_played,wins,losses,avg_points_scored,avg_points_allowed,avg_assists,avg_rebounds,...,avg_blocks,avg_fg_pct,avg_3pt_pct,avg_ft_pct,avg_bench_points,avg_paint_points,avg_fast_break_pts,avg_2nd_chance_pts,avg_biggest_lead,avg_plus_minus
0,15016,United,2025,1,0.0,1,97.000000,107.000000,23.000000,48.000000,...,1.000000,0.396000,0.293000,0.650000,39.000000,38.000000,7.0,12.0,1.000000,-10.000000
1,15018,Loong-Lions,2025,3,0.0,3,85.666667,131.666667,16.333333,26.666667,...,2.666667,0.362667,0.296667,0.822333,35.333333,22.666667,7.0,8.0,1.333333,-46.000000
2,50013,Phoenix,2025,1,0.0,1,92.000000,127.000000,24.000000,42.000000,...,6.000000,0.340000,0.174000,0.700000,39.000000,48.000000,5.0,21.0,0.000000,-35.000000
3,50014,Jerusalem B.C.,2025,1,0.0,1,88.000000,123.000000,18.000000,38.000000,...,2.000000,0.364000,0.207000,0.703000,33.000000,34.000000,6.0,10.0,2.000000,-35.000000
4,1610612737,Blackhawks,2049,19,5.0,14,78.736842,83.421053,NaN,NaN,...,NaN,0.333000,NaN,0.689105,NaN,NaN,NaN,NaN,NaN,-4.684211


In [6]:
summary = pd.DataFrame({
    "feature": per_season.columns,
    "dtype": per_season.dtypes.astype(str).values,
    "non_null": per_season.notna().sum().values,
    "missing_pct": (per_season.isna().mean() * 100).round(2).values,
})

num_cols = per_season.select_dtypes(include="number").columns
summary["min"] = np.nan
summary["max"] = np.nan
summary.loc[summary["feature"].isin(num_cols), "min"] = per_season[num_cols].min().values
summary.loc[summary["feature"].isin(num_cols), "max"] = per_season[num_cols].max().values


def infer_value_type(col, mn, mx, dtype):
    c = col.lower()

    if c.endswith("id"):
        return "identifier (discrete)"

    if c == "season":
        return "season (discrete integer)"

    if c in {"wins", "losses", "games_played"} or "count" in c:
        return "count (discrete)"

    if "pct" in c:
        if pd.notna(mx) and mx <= 1.5:
            return "percentage (0~1)"
        return "percentage (0~100)"

    if "plus_minus" in c:
        return "signed metric (negative to positive)"

    if pd.notna(mn) and pd.notna(mx) and mn >= -1 and mx <= 1:
        return "bounded ratio (-1~1)"

    if "int" in dtype:
        return "discrete integer"

    if "float" in dtype:
        return "continuous"

    return "categorical/text"


def source_from_feature(feature):
    f = feature.lower()

    if f in {"teamid", "teamname", "season", "games_played", "wins", "losses"}:
        return "team/season context"
    if "allowed" in f:
        return "defense"
    if any(k in f for k in ["assists", "rebounds", "turnovers", "steals", "blocks"]):
        return "playmaking/hustle"
    if any(k in f for k in ["fg_pct", "3pt_pct", "ft_pct"]):
        return "shooting efficiency"
    if any(k in f for k in ["bench", "paint", "fast_break", "2nd_chance"]):
        return "scoring profile"
    if any(k in f for k in ["points", "plus_minus", "lead"]):
        return "scoring & margin"
    return "other"


summary["value_type"] = summary.apply(
    lambda row: infer_value_type(row["feature"], row["min"], row["max"], row["dtype"]),
    axis=1,
)
summary["source"] = summary["feature"].map(source_from_feature)

summary = summary[[
    "source", "feature", "dtype", "value_type", "non_null", "missing_pct", "min", "max"
]].sort_values(["source", "feature"]).reset_index(drop=True)

print(f"Feature summary shape: {summary.shape}")
summary

Feature summary shape: (22, 8)


,source,feature,dtype,value_type,non_null,missing_pct,min,max
0,defense,avg_points_allowed,float64,continuous,1698,0.00,29.500000,1.316667e+02
1,playmaking/hustle,avg_assists,float64,continuous,1365,19.61,0.000000,5.300000e+01
2,playmaking/hustle,avg_blocks,float64,continuous,1308,22.97,0.000000,2.100000e+01
3,playmaking/hustle,avg_rebounds,float64,continuous,1387,18.32,0.000000,9.500000e+01
4,playmaking/hustle,avg_steals,float64,continuous,1300,23.44,0.000000,2.400000e+01
5,playmaking/hustle,avg_turnovers,float64,continuous,1296,23.67,0.000000,2.800000e+01
6,scoring & margin,avg_biggest_lead,float64,continuous,75,95.58,0.000000,2.069091e+01
7,scoring & margin,avg_plus_minus,float64,signed metric (negative to positive),1698,0.00,-46.000000,1.258333e+01
8,scoring & margin,avg_points_scored,float64,continuous,1698,0.00,32.000000,1.262235e+02
9,scoring profile,avg_2nd_chance_pts,float64,continuous,75,95.58,2.333333,2.100000e+01
